# Proposed adaptation and application export

I want to test whether adaptation adds value beyond the developed prompting baseline. A model that has been fine-tuned is not automatically more helpful or safer. The comparison needs matched inference prompts, reviewed evaluation data and a separate deployment assessment.

This legacy notebook is unexecuted in the recorded evidence and must not be run as part of Milestone 2. Its learning-rate exploration, training and export stages are proposed tooling. Earlier runtime estimates and claims that the base model already works well were not supported by measurements here. The Experiment 01 protocol defines the immediate baseline work and the gates before adaptation.

Milestone 2: prompt selection uses dev only. Few-shot conversations come from reserved official training groups. Formerly exposed validation groups are quarantined. Run scripts/check_leakage.py before generation. These legacy notebooks do not substitute for the revision-pinned Experiment 01 harness. No training is authorized in Milestone 2.


## 1. Setup
Upload `relevia-research` to Google Drive first, or clone it from GitHub. Accept Meta's licence on https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct before logging in.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/relevia-research
!pip install -q -r requirements.txt

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # use a token with WRITE access if you want step 6 to upload

## 2. Data boundaries

The intended training source is EmpatheticDialogues, which supports studying responses to emotional situations rather than establishing clinical competence. Development conversations support prompting, validation supports model selection, and final test conversations remain isolated. The leakage report records the earlier contamination and the correction; source permissions and privacy still require review.

In [ ]:
!python data/build_dataset.py

In [ ]:
MODEL = 'configs/models/llama-3.2-3b.yaml'

## 3. Find the learning rate
Short runs at 5e-5, 1e-4, 2e-4 and 3e-4. At the end it prints which one to use. **Put that number in `configs/base.yaml` → `train.lr`** before step 4 (it's 2e-4 right now).

In [ ]:
!python src/lr_sweep.py --model {MODEL}

## 4. Deferred adaptation

The working hypothesis is that adaptation may improve contextual quality beyond the tuned prompt, but it may also encourage formulaic reassurance or harmful agreement. LoRA represents a weight update as W′ = W + (alpha/r)BA, training low-rank factors while retaining the base weights. QLoRA additionally quantises the frozen base to reduce memory requirements. Fewer trainable parameters do not establish better dialogue. See the [literature notes](../research/notes/LITERATURE_REVIEW.md) for references and verification limits. The later comparison must control data, inference prompt and decoding, and record failures as well as loss.

In [ ]:
!python src/train.py --model {MODEL} --seed 1

## 5. Proposed exploratory comparison
Compare it with the plain model using the app-style prompt on 40 development messages. Open `results/side_by_side_tuned_prompt_vs_finetuned_tuned.md`, choose the better reply for each pair, *then* open the key.

The earlier suggestion to ship after 25 wins out of 40 is withdrawn as an unsupported release threshold. This exploratory comparison cannot substitute for a frozen test evaluation, uncertainty estimates and a safety review. Do not deploy on its basis.

In [ ]:
!python src/generate.py --model {MODEL} --condition tuned_prompt --testset dev --limit 40
!python src/generate.py --model {MODEL} --condition finetuned_tuned --testset dev --limit 40
!python src/side_by_side.py --model llama-3.2-3b --a tuned_prompt --b finetuned_tuned --testset dev

In [ ]:
# also check it didn't get worse at safety
!python src/generate.py --model {MODEL} --condition finetuned_tuned --testset safety/crisis
!python src/generate.py --model {MODEL} --condition tuned_prompt --testset safety/crisis
!python src/evaluate.py
import pandas as pd; pd.read_csv('results/metrics/auto_metrics.csv')[['condition','testset','crisis_referral_rate','false_alarm_rate']].dropna()

## 6. Deferred export investigation

Export introduces another source of change: merging and conversion can alter numerical behaviour or fail compatibility checks. Reassess quality and safety after conversion and measure the browser runtime on identified hardware. The commands below have not been validated as a release process, and their upload step requires a separately authorised publication decision.
Replace `<you>` with your Hugging Face username. At the end it prints a `CUSTOM_MODEL` block. Paste it into `relevia/src/services/modelConfig.js`.

In [ ]:
!pip install -q --pre -U -f https://mlc.ai/wheels mlc-llm-nightly-cu123 mlc-ai-nightly-cu123
!python deploy/export_to_webllm.py --model {MODEL} --seed 1 --hf-repo <you>/Relevia-3B-ft-q4f16_1-MLC

## What remains unresolved
No executed outputs in this notebook establish an improvement in dialogue quality, safety or deployment performance. The next useful evidence is a controlled baseline run after the Milestone 2 gates are resolved. Preserve unsuccessful runs and interpret any later results against the recorded controls, rather than treating a completed cell as a successful experiment.
